[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hashirama21/neoplasia-detection/blob/main/RARE2026.ipynb)

# RARE2026 — Barrett's Neoplasia Detection

**Task:** Binary classification (neoplasia vs. NDBE) on endoscopy images.  
**Metric:** PPV @ 90% Recall — bootstrap-simulated at 1% clinical prevalence.  
**Strategy:** DINOv2 ViT-B/14 + GastroNet-5M weights · LoRA (rank=8) · Asymmetric Loss · Isotonic calibration.

| Section | Content |
|---|---|
| 1 | BONSAI dataset — download, EDA, patient-aware splits |
| 2 | EVC Barretts FullSet — structure, masks, texture analysis |
| 3 | Dataset integration — merge train + enrich val_calibration |
| 4 | Repository setup, dependencies, GastroNet weights |
| 5 | Training — 5 independent seeds (true ensemble) |
| 6 | Calibration + bootstrap evaluation |
| 7 | Inference demo |

In [ ]:
# ── peft MUST be installed before any model import ─────────────────────────
# Restart the runtime after this cell if running for the first time.
%pip install peft -q

import importlib
if importlib.util.find_spec('peft') is None:
    raise RuntimeError('peft installation failed — re-run this cell or restart the runtime.')

import peft
print(f'✓ peft {peft.__version__} — LoRA will be active')

In [ ]:
# ── Runtime constants — all paths use /root/ (Modal runtime) ───────────────
REPO_DIR    = '/root/rare26'
DATA_DIR    = '/root/data'
EVC_DIR     = '/root/EVC_Barretts_FullSet'
WEIGHTS_DIR = '/root/rare26/weights'
OUTPUT_DIR  = '/root/outputs'

BONSAI_ZIP     = '/root/bonsai_dataset.zip'
BONSAI_EXTRACT = '/root/bonsai_raw'

import os, torch
for d in [DATA_DIR, WEIGHTS_DIR, OUTPUT_DIR]:
    os.makedirs(d, exist_ok=True)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
if device == 'cuda':
    name = torch.cuda.get_device_name(0)
    mem  = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU : {name}  ({mem:.1f} GB)')
else:
    print('No GPU detected — check runtime settings.')

print(f'\nPaths:')
for k, v in [('REPO', REPO_DIR), ('DATA', DATA_DIR), ('EVC', EVC_DIR),
             ('WEIGHTS', WEIGHTS_DIR), ('OUTPUT', OUTPUT_DIR)]:
    print(f'  {k:<8}: {v}')

---
## Section 1 — BONSAI Dataset (Official RARE26 Training Data)

3 095 images from 2 endoscopy centres (center_1 / center_2).  
158 neoplasia (neo) · 2 937 NDBE.  
Filenames are hashes — no visible patient ID.

In [ ]:
import os, zipfile, gdown

BONSAI_FILE_ID = '1Hs9O6Gckq3CUuPMN5Symq5roQahreYnx'

if not os.path.exists(BONSAI_EXTRACT):
    print('Downloading BONSAI dataset...')
    gdown.download(f'https://drive.google.com/uc?id={BONSAI_FILE_ID}', BONSAI_ZIP, quiet=False)
    os.makedirs(BONSAI_EXTRACT, exist_ok=True)
    with zipfile.ZipFile(BONSAI_ZIP, 'r') as z:
        z.extractall(BONSAI_EXTRACT)
    print('Extraction complete.')
else:
    print('BONSAI data already extracted.')

for root, dirs, files in os.walk(BONSAI_EXTRACT):
    imgs = [f for f in files if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    if imgs:
        rel = os.path.relpath(root, BONSAI_EXTRACT)
        print(f'  {rel:<40} {len(imgs):>5} images')

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from pathlib import Path

EXTS = {'.jpg', '.jpeg', '.png'}

rows = []
for root, _, files in os.walk(BONSAI_EXTRACT):
    imgs = [f for f in files if Path(f).suffix.lower() in EXTS]
    if not imgs: continue
    parts = Path(root).relative_to(BONSAI_EXTRACT).parts
    center  = parts[0] if len(parts) > 0 else 'unknown'
    cls_raw = parts[1] if len(parts) > 1 else 'unknown'
    for f in imgs:
        rows.append({'image_path': os.path.join(root, f), 'center': center, 'class_raw': cls_raw})

bonsai_df = pd.DataFrame(rows)
NEG_CLASS = 'ndbe'
bonsai_df['label'] = (bonsai_df['class_raw'] != NEG_CLASS).astype(int)

print(f'Total images : {len(bonsai_df)}')
print(bonsai_df.groupby(['center', 'class_raw'])['label'].count().to_string())

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.countplot(data=bonsai_df, x='center', hue='class_raw', ax=axes[0])
axes[0].set_title('Images per centre')
sns.countplot(data=bonsai_df, x='class_raw', ax=axes[1], palette=['steelblue', 'tomato'])
axes[1].set_title('Class balance  (pos=neo)')
plt.tight_layout()
plt.show()

In [ ]:
sample = bonsai_df.sample(min(200, len(bonsai_df)), random_state=42)

size_rows = []
for _, row in sample.iterrows():
    try:
        with Image.open(row['image_path']) as img:
            w, h = img.size
            size_rows.append({'class_raw': row['class_raw'], 'width': w, 'height': h})
    except Exception:
        continue

sizes_df = pd.DataFrame(size_rows)
print('Image dimension statistics:')
print(sizes_df.groupby('class_raw')[['width', 'height']].describe().T.to_string())

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.scatterplot(data=sizes_df, x='width', y='height', hue='class_raw', alpha=0.5, ax=axes[0])
axes[0].set_title('Resolution scatter')
for col, color in [('width', 'royalblue'), ('height', 'tomato')]:
    sizes_df[col].plot(kind='hist', bins=20, alpha=0.5, color=color, label=col, ax=axes[1])
axes[1].legend(); axes[1].set_title('Width / Height distributions')
plt.tight_layout(); plt.show()

print(f"\nMedian size: {sizes_df['width'].median():.0f} x {sizes_df['height'].median():.0f} px")
print('392px crop → 28x28 DINOv2 patches at patch_size=14.')

### 1.3 — Train / val split

BONSAI filenames are hashes — no patient ID extractable. `StratifiedShuffleSplit` on images.
EVC data (Section 2) uses `GroupShuffleSplit` by `patient_id` from its `patXX_imY_DIAGNOSIS` filenames.

In [ ]:
from sklearn.model_selection import StratifiedShuffleSplit

bonsai_df[['image_path', 'label']].to_csv(f'{DATA_DIR}/bonsai_all.csv', index=False)

sss1 = StratifiedShuffleSplit(n_splits=1, test_size=0.15, random_state=42)
train_idx, val_idx = next(sss1.split(np.zeros(len(bonsai_df)), bonsai_df['label'].values))
train_df = bonsai_df.iloc[train_idx][['image_path', 'label']].reset_index(drop=True)
val_df   = bonsai_df.iloc[val_idx][['image_path', 'label']].reset_index(drop=True)

sss2 = StratifiedShuffleSplit(n_splits=1, test_size=0.30, random_state=42)
sel_idx, cal_idx = next(sss2.split(np.zeros(len(val_df)), val_df['label'].values))
val_sel_df = val_df.iloc[sel_idx].reset_index(drop=True)
val_cal_df = val_df.iloc[cal_idx].reset_index(drop=True)

train_df.to_csv(f'{DATA_DIR}/train.csv',             index=False)
val_df.to_csv(f'{DATA_DIR}/val.csv',                 index=False)
val_sel_df.to_csv(f'{DATA_DIR}/val_selection.csv',   index=False)
val_cal_df.to_csv(f'{DATA_DIR}/val_calibration.csv', index=False)

print('BONSAI splits written:')
for name, df in [('train', train_df), ('val_selection', val_sel_df), ('val_calibration', val_cal_df)]:
    print(f'  {name:<18}: {len(df):5d} images  pos={df["label"].sum():3d}  neg={(df["label"]==0).sum():4d}')

---
## Section 2 — EVC Barretts FullSet

External dataset from a previous TU/e challenge.  
Used to **enrich val_calibration** (target ≥ 50 positives) and expand training.

**Structure:** `patXX_imY_DIAGNOSIS.{png,bmp}` — patient ID is explicit in the filename.  
**Labels:** `ACHD` = neoplasia (label=1) · `NDBT` = normal (label=0).  
**Annotations:** per-pixel expert masks in BMP format (5 annotators).

**Risk:** ACHD lesions have ~17% image coverage (visible lesions).  
RARE26 test targets subtle lesions at <1% prevalence.  
Effect on subtle cases measurable only after Open Phase submission.

**Expected path:** `EVC_DIR = /root/EVC_Barretts_FullSet`  
Extract the ZIP before running this section:
```python
import zipfile
with zipfile.ZipFile('/root/EVC_Barretts_FullSet.zip') as z:
    z.extractall('/root/EVC_Barretts_FullSet')
```

In [ ]:
from pathlib import Path

if Path(EVC_DIR).exists():
    n_imgs = sum(1 for p in Path(EVC_DIR).rglob('*')
                 if p.suffix.lower() in {'.png', '.bmp', '.jpg'})
    print(f'EVC found at {EVC_DIR}  ({n_imgs} files)')
    EVC_AVAILABLE = True
else:
    print(f'EVC not found at {EVC_DIR}')
    print('Extract the ZIP first:')
    print("  import zipfile")
    print("  with zipfile.ZipFile('/root/EVC_Barretts_FullSet.zip') as z:")
    print(f"      z.extractall('{EVC_DIR}')")
    EVC_AVAILABLE = False

In [ ]:
if not EVC_AVAILABLE:
    print('Skip — EVC not available.')
else:
    import re, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
    from pathlib import Path

    file_data = []
    for root, _, files in os.walk(EVC_DIR):
        for f in files:
            ext  = Path(f).suffix.lower()
            stem = Path(f).stem
            parts = stem.split('_')
            if len(parts) >= 3:
                file_data.append({
                    'filepath':   os.path.join(root, f),
                    'patient_id': parts[0],
                    'image_id':   parts[1],
                    'diagnosis':  parts[2],
                    'file_type':  'mask' if ext == '.bmp' else 'image',
                    'extension':  ext,
                })

    evc_meta = pd.DataFrame(file_data)
    evc_imgs = evc_meta[evc_meta['file_type'] == 'image'].copy()

    print(f'EVC total files : {len(evc_meta)}')
    print(f'  Images        : {len(evc_imgs)}')
    print(f'  Masks (BMP)   : {(evc_meta["file_type"]=="mask").sum()}')
    print()
    print(evc_imgs['diagnosis'].value_counts().to_string())
    print()
    print(f'Unique patients : {evc_imgs["patient_id"].nunique()}')
    print(evc_imgs.groupby('diagnosis')['patient_id'].nunique().rename('unique patients').to_string())

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    sns.countplot(data=evc_imgs, x='diagnosis', palette=['steelblue', 'tomato'], ax=axes[0])
    axes[0].set_title('EVC — images per diagnosis')
    pc = evc_imgs.groupby(['patient_id', 'diagnosis']).size().reset_index(name='count')
    sns.histplot(data=pc, x='count', hue='diagnosis', bins=20, ax=axes[1])
    axes[1].set_title('Images per patient')
    plt.tight_layout(); plt.show()

In [ ]:
if not EVC_AVAILABLE:
    print('Skip.')
else:
    from PIL import Image
    fig, axes = plt.subplots(2, 4, figsize=(18, 9))
    for row_idx, diag in enumerate(['ACHD', 'NDBT']):
        samples = evc_imgs[evc_imgs['diagnosis'] == diag].sample(4, random_state=42)
        for col_idx, (_, meta) in enumerate(samples.iterrows()):
            ax = axes[row_idx][col_idx]
            try:
                ax.imshow(Image.open(meta['filepath']).convert('RGB'))
            except Exception as e:
                ax.text(0.5, 0.5, str(e), ha='center', va='center', fontsize=8)
            ax.set_title(f"{diag} | {meta['patient_id']}", fontsize=9)
            ax.axis('off')
    plt.suptitle('EVC — ACHD vs NDBT', fontsize=13)
    plt.tight_layout(); plt.show()

In [ ]:
if not EVC_AVAILABLE:
    print('Skip.')
else:
    import cv2, numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns

    evc_masks = evc_meta[evc_meta['file_type'] == 'mask']
    mask_stats = []
    for _, row in evc_masks.sample(min(150, len(evc_masks)), random_state=42).iterrows():
        mask = cv2.imread(row['filepath'], cv2.IMREAD_GRAYSCALE)
        if mask is not None:
            mask_stats.append({
                'diagnosis':   row['diagnosis'],
                'coverage_pct': float((mask > 0).mean() * 100),
            })

    df_masks = pd.DataFrame(mask_stats)
    print('Expert annotation coverage (% image area):')
    print(df_masks.groupby('diagnosis')['coverage_pct'].describe().round(2).to_string())

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    sns.violinplot(data=df_masks, x='diagnosis', y='coverage_pct', inner='box',
                   palette=['steelblue', 'tomato'], ax=axes[0])
    axes[0].set_title('Lesion coverage % — expert masks')
    achd_cov = df_masks[df_masks['diagnosis'] == 'ACHD']['coverage_pct']
    axes[1].hist(achd_cov, bins=20, color='tomato', edgecolor='white')
    axes[1].axvline(achd_cov.median(), color='black', linestyle='--',
                    label=f'Median {achd_cov.median():.1f}%')
    axes[1].set_title('ACHD coverage distribution'); axes[1].legend()
    plt.tight_layout(); plt.show()
    print(f'ACHD median coverage = {achd_cov.median():.1f}%  '
          '(RARE26 test has much subtler lesions — verify on test set)')

In [ ]:
if not EVC_AVAILABLE:
    print('Skip.')
else:
    from skimage.feature import local_binary_pattern
    from skimage.color import rgb2gray
    import cv2, numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns

    evc_png = evc_imgs[evc_imgs['extension'] == '.png']
    sampled = evc_png.groupby('diagnosis').sample(min(50, len(evc_png)//2), random_state=42)

    texture_rows = []
    for _, row in sampled.iterrows():
        img = cv2.imread(row['filepath'])
        if img is None: continue
        gray = rgb2gray(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        lbp  = local_binary_pattern(gray, 24, 3, method='uniform')
        texture_rows.append({
            'diagnosis':        row['diagnosis'],
            'texture_variance': float(np.var(lbp)),
            'brightness_std':   float(np.std(gray)),
        })

    df_tex = pd.DataFrame(texture_rows)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    sns.kdeplot(data=df_tex, x='texture_variance', hue='diagnosis', fill=True, ax=axes[0])
    axes[0].set_title('Texture variance (LBP)')
    sns.boxplot(data=df_tex, x='diagnosis', y='brightness_std',
                palette=['steelblue', 'tomato'], ax=axes[1])
    axes[1].set_title('Brightness heterogeneity')
    plt.tight_layout(); plt.show()
    print('DINOv2 GastroNet-5M captures these patterns at all scales.'
          ' LBP/GLCM add noise, not signal, for this backbone.')

---
## Section 3 — Dataset Integration

Objective: **≥ 50 positives in val_calibration** for continuous bootstrap recall.
EVC ACHD patients are split by `patient_id` — no patient leakage between train and calibration.

In [ ]:
if not EVC_AVAILABLE:
    print('EVC not available — using BONSAI-only splits.')
    print(f'val_calibration has ~7 positives → metrics will have high variance.')
    TRAIN_CSV   = f'{DATA_DIR}/train.csv'
    VAL_CAL_CSV = f'{DATA_DIR}/val_calibration.csv'
else:
    !python {REPO_DIR}/scripts/prepare_evc.py \
        --evc_dir {EVC_DIR} \
        --train_csv {DATA_DIR}/train.csv \
        --val_cal_csv {DATA_DIR}/val_calibration.csv \
        --out_dir {DATA_DIR} \
        --target_cal_pos 50 \
        --seed 42

    import pandas as pd
    TRAIN_CSV   = f'{DATA_DIR}/train_merged.csv'
    VAL_CAL_CSV = f'{DATA_DIR}/val_calibration_enriched.csv'

    for name, path in [('train_merged', TRAIN_CSV),
                       ('val_selection', f'{DATA_DIR}/val_selection.csv'),
                       ('val_calibration_enriched', VAL_CAL_CSV)]:
        df = pd.read_csv(path)
        pos = df['label'].sum()
        neg = (df['label'] == 0).sum()
        print(f'  {name:<28}: {len(df):5d} images  pos={pos:3d}  neg={neg:4d}')

    n_pos = pd.read_csv(VAL_CAL_CSV)['label'].sum()
    if n_pos < 35:
        print(f'\n⚠ val_calibration has only {n_pos} positives — bootstrap still unreliable.')
    else:
        print(f'\n✓ val_calibration has {n_pos} positives — bootstrap will be interpretable.')

---
## Section 4 — Repository Setup, Dependencies, GastroNet Weights

In [ ]:
import os, sys
from pathlib import Path

if not Path(REPO_DIR).exists():
    !git clone https://github.com/hashirama21/neoplasia-detection.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull origin main

sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)
print(f'Working directory: {os.getcwd()}')

In [ ]:
%pip install -r {REPO_DIR}/requirements.txt -q

# Verify peft (already installed in cell 1 — this is a sanity check)
import peft
print(f'✓ peft {peft.__version__} confirmed')

In [ ]:
from pathlib import Path
import torch, timm

WEIGHTS_PATH = Path(WEIGHTS_DIR) / 'dinov2_gastronet5m.pth'
Path(WEIGHTS_DIR).mkdir(parents=True, exist_ok=True)

if WEIGHTS_PATH.exists():
    print(f'Weights already present: {WEIGHTS_PATH}')
else:
    # Strategy 1: GastroNet-5M (endoscopy domain pretraining)
    try:
        from huggingface_hub import hf_hub_download
        print('Downloading GastroNet-5M from HuggingFace...')
        hf_hub_download(
            repo_id='BONS-AI-TUE-AMC/GastroNetDinov2',
            filename='dinov2_gastronet5m.pth',
            local_dir=WEIGHTS_DIR,
        )
        print('GastroNet-5M downloaded.')
    except Exception as e:
        print(f'GastroNet-5M unavailable ({e}). Falling back to ImageNet DINOv2...')
        # Strategy 2: ImageNet DINOv2 via timm
        backbone = timm.create_model('vit_base_patch14_dinov2.lvd142m', pretrained=True, num_classes=0)
        torch.save(backbone.state_dict(), WEIGHTS_PATH)
        print('ImageNet DINOv2 weights saved.')

if WEIGHTS_PATH.exists():
    ckpt  = torch.load(WEIGHTS_PATH, map_location='cpu', weights_only=True)
    state = ckpt if not any(k in ckpt for k in ('model_state', 'state_dict', 'model')) else \
            (ckpt.get('model_state') or ckpt.get('state_dict') or ckpt.get('model') or ckpt)
    n_par = sum(v.numel() for v in state.values() if isinstance(v, torch.Tensor))
    print(f'Checkpoint: {WEIGHTS_PATH.stat().st_size/1e6:.1f} MB | {n_par/1e6:.1f}M params')
    print(f'Keys sample: {list(state.keys())[:4]}')
else:
    print('No weights found — training will use random initialization.')

---
## Section 5 — Training (5-seed ensemble)

5 **independent runs** with different seeds — not 5 checkpoints of the same run.  
Ensemble = mean logits over all 5 runs after common calibration.

| Config | Value |
|---|---|
| Epochs | 30 |
| Image size | 392 px (28×28 DINOv2 patches, patch_size=14) |
| Backbone LR | 1e-5 (LoRA adapters only) |
| Head LR | 1e-3 |
| Loss | AsymmetricLoss γ⁻=4, γ⁺=1, clip=0.05 |
| Scheduler | CosineAnnealing, T_max=epochs |
| Seeds | 42, 123, 456, 789, 1337 |

**Estimated time per run:** ~7 min (H100) · ~15 min (A10G) · ~25 min (T4)

In [ ]:
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# Fallback if Section 3 was skipped
try:
    TRAIN_CSV
except NameError:
    TRAIN_CSV   = f'{DATA_DIR}/train.csv'
    VAL_CAL_CSV = f'{DATA_DIR}/val_calibration.csv'

SEEDS = [42, 123, 456, 789, 1337]

for seed in SEEDS:
    print(f'\n{"="*60}')
    print(f'Training  seed={seed}')
    print(f'{"="*60}')

    seed_output = f'{OUTPUT_DIR}/seed_{seed}'

    !python {REPO_DIR}/scripts/train.py \
        project.output_dir={seed_output} \
        project.seed={seed} \
        paths.data_dir={DATA_DIR} \
        paths.weights_dir={WEIGHTS_DIR} \
        device={device} \
        num_workers=2 \
        pin_memory=True \
        training.epochs=30 \
        training.cross_validation.enabled=false \
        training.loss.gamma_neg=4 \
        data.image_size=392 \
        data.augmentation.val.center_crop=392 \
        data.train_csv={TRAIN_CSV} \
        data.val_calibration_csv={VAL_CAL_CSV} \
        data.batch_size=12 \
        data.oversample_factor=3.0 \
        data.pos_weight_factor=18.6 \
        model.backbone.img_size=392 \
        model.lora.enabled=true

print('\nAll 5 training runs complete.')

In [ ]:
import json, glob, pandas as pd
from pathlib import Path
from IPython.display import display

rows = []
for seed in SEEDS:
    cal_file = Path(OUTPUT_DIR) / f'seed_{seed}' / 'results' / 'calibration_results.json'
    if cal_file.exists():
        with open(cal_file) as f:
            r = json.load(f)
        rows.append({
            'seed':          seed,
            'median_ppv':    r.get('median_ppv', float('nan')),
            'median_recall': r.get('median_recall', float('nan')),
            'std_ppv':       r.get('std_ppv', float('nan')),
            'threshold':     r.get('optimal_threshold', float('nan')),
        })
    else:
        rows.append({'seed': seed, 'status': 'no results yet'})

df_runs = pd.DataFrame(rows)
print('Training summary (per-seed calibration results):')
display(df_runs)

---
## Section 6 — Calibration & Bootstrap Evaluation

Evaluation on **val_selection** only (val_calibration was used for threshold — never mix).

Stability criteria for Docker submission:
- PPV std < 0.15 → calibration stable
- Median recall ≥ 0.88 → recall constraint met with margin
- P10 PPV > 0.30 → correct behaviour even in worst bootstrap samples

**Run 1 results (BONSAI-only, no LoRA, 20 epochs, val_selection ~3 positives):**

| Metric | Value | Interpretable? |
|---|---|---|
| Median PPV@90Recall | 1.0000 | No — only 3 positives in val_selection |
| Std PPV | 0.1043 | No — sample too small |
| Median Recall | 0.6667 | No — at fixed threshold 0.5010 |

These results are not interpretable at this sample size.  
EVC integration (Section 3) is required for reliable evaluation.

In [ ]:
!python {REPO_DIR}/scripts/evaluate.py \
    project.output_dir={OUTPUT_DIR} \
    paths.data_dir={DATA_DIR} \
    paths.weights_dir={WEIGHTS_DIR} \
    device={device} \
    num_workers=2 \
    pin_memory=True

In [ ]:
import json, pandas as pd
from pathlib import Path
from IPython.display import display

eval_path = Path(OUTPUT_DIR) / 'results' / 'evaluation_results.json'
if not eval_path.exists():
    eval_path = Path(OUTPUT_DIR) / 'seed_42' / 'results' / 'evaluation_results.json'

if eval_path.exists():
    with open(eval_path) as f:
        ev = json.load(f)

    metrics = {
        'Median PPV@90Recall': ev.get('median_ppv', ev.get('median_ppv_at_90recall', 'N/A')),
        'Mean PPV':            ev.get('mean_ppv', 'N/A'),
        'Std PPV':             ev.get('std_ppv', 'N/A'),
        'P10 PPV':             ev.get('p10_ppv', 'N/A'),
        'P90 PPV':             ev.get('p90_ppv', 'N/A'),
        'Median Recall':       ev.get('median_recall', 'N/A'),
        'Threshold':           ev.get('optimal_threshold', ev.get('threshold_used', 'N/A')),
    }

    df_ev = pd.DataFrame(metrics.items(), columns=['Metric', 'Value'])
    df_ev['Value'] = df_ev['Value'].apply(lambda x: f'{x:.4f}' if isinstance(x, float) else str(x))
    display(df_ev.set_index('Metric'))

    std = ev.get('std_ppv', 999)
    rec = ev.get('median_recall', 0)
    p10 = ev.get('p10_ppv', 0)

    checks = [
        (std < 0.15,  f'PPV std < 0.15        : {std:.4f}'),
        (rec >= 0.88, f'Median recall >= 0.88  : {rec:.4f}'),
        (p10 > 0.30,  f'P10 PPV > 0.30        : {p10:.4f}'),
    ]

    print('\nStability criteria for Docker submission:')
    all_pass = True
    for passed, label in checks:
        print(f'  {"✓" if passed else "✗"} {label}')
        if not passed: all_pass = False

    print('\n✓ Ready for submission.' if all_pass else '\n✗ Not ready — investigate before submitting.')
else:
    print(f'No results at {eval_path} — run evaluate cell first.')

---
## Section 7 — Inference Demo

Run the best checkpoint on 4 random images.

In [ ]:
import glob, json, random, sys
import numpy as np
import matplotlib.pyplot as plt
import torch
import torchvision.transforms as T
from pathlib import Path
from PIL import Image

sys.path.insert(0, REPO_DIR)
from src.models.rare26_model import Rare26Model
from src.calibration.calibrator import IsotonicCalibrator
from omegaconf import OmegaConf

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

all_ckpts = sorted(
    glob.glob(f'{OUTPUT_DIR}/**/checkpoints/*.pt', recursive=True)
    + glob.glob(f'{OUTPUT_DIR}/checkpoints/*.pt'),
    key=lambda p: float(Path(p).stem.rsplit('_', 1)[-1]),
    reverse=True,
)

if not all_ckpts:
    print('No checkpoint — run Section 5 first.')
else:
    best_ckpt = all_ckpts[0]
    print(f'Loading: {Path(best_ckpt).name}')

    model_cfg = OmegaConf.load(f'{REPO_DIR}/configs/model/dinov2_gastronet.yaml')
    OmegaConf.update(model_cfg, 'checkpoint_path', f'{WEIGHTS_DIR}/dinov2_gastronet5m.pth')
    OmegaConf.update(model_cfg, 'backbone.img_size', 392)
    model = Rare26Model(model_cfg).to(device)
    ckpt = torch.load(best_ckpt, map_location=device, weights_only=True)
    model.load_state_dict(ckpt['model_state'])
    model.eval()

    cal_pkl  = Path(OUTPUT_DIR) / 'seed_42' / 'results' / 'isotonic_calibrator.pkl'
    cal_json = Path(OUTPUT_DIR) / 'seed_42' / 'results' / 'calibration_results.json'
    calibrator = None
    threshold  = 0.5
    if cal_pkl.exists():
        calibrator = IsotonicCalibrator()
        calibrator.load(str(cal_pkl))
    if cal_json.exists():
        with open(cal_json) as f:
            threshold = json.load(f).get('optimal_threshold', 0.5)
    print(f'Calibrator: {"loaded" if calibrator else "not found"}  |  Threshold: {threshold:.4f}')

    # Transform must match training resolution
    transform = T.Compose([
        T.Resize(448, interpolation=T.InterpolationMode.BICUBIC),
        T.CenterCrop(392),
        T.ToTensor(),
        T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])

    all_imgs = (
        glob.glob(f'{BONSAI_EXTRACT}/**/*.jpg', recursive=True) +
        glob.glob(f'{BONSAI_EXTRACT}/**/*.png', recursive=True)
    )
    samples = random.sample(all_imgs, min(4, len(all_imgs)))

    fig, axes = plt.subplots(1, len(samples), figsize=(16, 4))
    if len(samples) == 1: axes = [axes]

    for ax, img_path in zip(axes, samples):
        img = Image.open(img_path).convert('RGB')
        with torch.no_grad():
            logit = model(transform(img).unsqueeze(0).to(device)).squeeze().item()
        raw_prob = 1.0 / (1.0 + np.exp(-logit))
        cal_prob = float(calibrator.transform(np.array([raw_prob]))[0]) if calibrator else raw_prob
        pred     = cal_prob >= threshold

        parent   = Path(img_path).parent.name.lower()
        true_cls = 'neo' if parent not in ('ndbe', 'ndbt') else 'ndbe'

        ax.imshow(img)
        ax.set_title(
            f'True: {true_cls}\nProb: {cal_prob:.3f} → {"NEOPLASIA" if pred else "NDBE"}',
            color='red' if pred else 'green', fontsize=9,
        )
        ax.axis('off')

    plt.suptitle(f'Inference demo — threshold: {threshold:.3f}', fontsize=12)
    plt.tight_layout()
    plt.show()